# BTP final Kaggle pipeline — Mamba, five superclasses and 23 subclasses
Primary execution notebook. Configure **only section 1**, attach sources/data/wheels, then Run All / Save & Run All.
No writable source-copy patches, interactive prompts, manual reloads or cell reruns are required in a fresh session.
`04_btp_experiments.ipynb` and `ecg-xl-btp.ipynb` remain development/attempt references.

The user reports real CUDA/AMP smoke success on PTB-XL **1.0.1**, Python 3.13.15,
Torch 2.11.0+cu128, T4, ABI=True; this notebook does not claim completed GPU benchmarks.
Default settings start both 10-epoch experiments. Use DIAGNOSTIC_MODE=True for a short batch-limited check,
which never produces held-out test or benchmark results. Models run sequentially.

## 1. Central configuration — all execution choices live here

In [ ]:
import os
from pathlib import Path

RUN_SUPERCLASS = True
RUN_SUBCLASS = True
RUN_SMOKE = True
RUN_METADATA_AUDIT = True
FULL_EPOCHS = 10
SUPERCLASS_EPOCHS = None  # None uses FULL_EPOCHS
SUBCLASS_EPOCHS = None
FULL_BATCH_SIZE = 8
SMOKE_BATCH_SIZE = 2
DIAGNOSTIC_MODE = False
DIAGNOSTIC_BATCHES = 2  # train and validation only, per epoch; no test evaluation

SOURCE_ROOT = os.environ.get("BTP_REPO_DIR", "")  # auto-discover if empty
DATASET_ROOT = os.environ.get("PTBXL_DATA_DIR", "")
DATASET_VERSION = os.environ.get("PTBXL_DATASET_VERSION", "1.0.1")  # operator-declared; match YOUR attached input
OUTPUT_ROOT = "/kaggle/working/btp_outputs" if Path("/kaggle/working").is_dir() else "outputs/final"
CACHE_DIR = "/kaggle/working/btp_cache" if Path("/kaggle/working").is_dir() else "outputs/ptbxl_cache"
NORMALIZATION_MODE = "sampled"  # sampled (fast) or full (exact)
NORMALIZATION_SAMPLES = 1000
NORMALIZATION_SEED = 42
CACHE_SIGNALS = True
NUM_WORKERS = 0  # reliable first-batch path; only increase after profiling
APPLY_FILTER = True
AUGMENT = True
MIXED_PRECISION = True
SEED = 42
LOG_EVERY_BATCHES = 20
MODEL_SETTINGS = dict(embed_dim=384, hidden_dim=384, n_layers=8, d_state=16,
                      d_conv=4, expand=2, dropout=0.45, bidirectional=True, pooling="mean_attention")

INSTALL_BASE = True  # auto-install missing base dependencies, not Torch
WHEEL_DIR = ""  # auto-find attached mamba_ssm/causal_conv1d wheels if empty
ALLOW_SOURCE_BUILD = True  # explicit fallback when working CUDA packages/wheels are unavailable
MAX_JOBS = 2
RESUME_RUN_DIRS = {"superclass": "", "subclass": ""}  # original run dir with last.pth AND best.pth
CONTINUE_OTHER_TASK_ON_FAILURE = True
ACTIVE = any((RUN_SUPERCLASS, RUN_SUBCLASS, RUN_SMOKE, RUN_METADATA_AUDIT))
NEED_GPU = any((RUN_SUPERCLASS, RUN_SUBCLASS, RUN_SMOKE))

## 2. Source/dataset discovery
Attach extracted source and PTB-XL folders. Ambiguous roots require an override in section 1;
no guessed credentials or GitHub downloads occur. Version is recorded as **operator-declared**, with
root and metadata hashes. The successful input was labeled 1.0.1; do not call it 1.0.3.

In [ ]:
import sys
from datetime import datetime, timezone
from uuid import uuid4
TASK_RESULTS, TASK_ERRORS, PREPARED, SMOKE_REPORTS = {}, {}, {}, {}
if ACTIVE:
    candidates = [Path(SOURCE_ROOT).resolve()] if SOURCE_ROOT else []
    if not SOURCE_ROOT:
        if (Path.cwd() / "src/kaggle_setup.py").is_file():
            candidates.append(Path.cwd().resolve())
        input_root = Path("/kaggle/input")
        if input_root.is_dir():
            candidates.extend(p.parent.parent for p in input_root.rglob("kaggle_setup.py")
                              if p.parent.name == "src" and (p.parent.parent / "05_btp_kaggle_final.ipynb").is_file())
    candidates = sorted(set(candidates))
    if len(candidates) != 1 or not (candidates[0] / "src/kaggle_setup.py").is_file():
        raise RuntimeError(f"Set SOURCE_ROOT to the extracted current repository; candidates={candidates}")
    REPO_DIR = candidates[0]
    sys.path.insert(0, str(REPO_DIR))
    from src.kaggle_setup import setup_dependencies, stage_log
    SESSION_DIR = Path(OUTPUT_ROOT) / "sessions" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8])
    SESSION_DIR.mkdir(parents=True, exist_ok=True)
    print("Source:", REPO_DIR, "Session logs:", SESSION_DIR, "Cache:", CACHE_DIR, flush=True)

## 3. Dependency setup and actual CUDA checks
Working installed packages are probed in a fresh interpreter and retained. Supplied compatible wheels are preferred;
otherwise ALLOW_SOURCE_BUILD explicitly enables pinned source wheels with KEEP_CUDA=True, MAX_JOBS=2,
--no-build-isolation and verbose logs. Source-built wheels are saved for future sessions.
Torch is never installed by this setup. Fresh sessions must reattach/install dependencies.
A small Mamba CUDA forward/backward probe must pass before expensive preprocessing.

In [ ]:
if ACTIVE:
    with stage_log(SESSION_DIR, "dependency setup and CUDA probe"):
        setup_report = (setup_dependencies(REPO_DIR, SESSION_DIR, wheel_dir=WHEEL_DIR,
            allow_source_build=ALLOW_SOURCE_BUILD, install_base=INSTALL_BASE, max_jobs=MAX_JOBS)
            if NEED_GPU else {"installation": "skipped_metadata_only"})
        import gc
        import torch
        from dataclasses import asdict, replace
        from src.config import DataConfig, ModelConfig, ExperimentConfig
        from src.data import load_ptbxl
        from src.kaggle import find_ptbxl_root, audit_metadata, require_cuda, cached_smoke_data, smoke_forward_backward
        from src.models.registry import create_model
        from src.experiments import run_experiment, diagnostic_limits
        from src.utils import environment_info, write_json, seed_everything, log_progress
        if NEED_GPU:
            require_cuda()
        DATA_DIR = find_ptbxl_root(DATASET_ROOT or None)
        print("Dataset:", DATA_DIR, "Declared release:", DATASET_VERSION, flush=True)
        write_json(SESSION_DIR / "environment.json", environment_info())
        write_json(SESSION_DIR / "setup_report.json", setup_report)
        DATA_CONFIG = DataConfig(data_dir=str(DATA_DIR), sampling_rate=100, duration=10,
            num_workers=NUM_WORKERS, apply_filter=APPLY_FILTER, augment=AUGMENT,
            dataset_version=DATASET_VERSION, normalization_mode=NORMALIZATION_MODE,
            normalization_samples=NORMALIZATION_SAMPLES, normalization_seed=NORMALIZATION_SEED,
            cache_dir=str(Path(CACHE_DIR).resolve()), cache_signals=CACHE_SIGNALS)
        CONFIG = ExperimentConfig(epochs=FULL_EPOCHS, batch_size=FULL_BATCH_SIZE, device="cuda",
            seed=SEED, mixed_precision=MIXED_PRECISION, model=ModelConfig(**MODEL_SETTINGS),
            data=DATA_CONFIG, diagnostic=DIAGNOSTIC_MODE, diagnostic_batches=DIAGNOSTIC_BATCHES,
            log_every_batches=LOG_EVERY_BATCHES, output_root=str(SESSION_DIR / "experiments"))
        write_json(SESSION_DIR / "session_config.json", {"experiment": asdict(CONFIG),
            "controls": {"superclass": RUN_SUPERCLASS, "subclass": RUN_SUBCLASS, "smoke": RUN_SMOKE,
                         "metadata_audit": RUN_METADATA_AUDIT},
            "task_epoch_overrides": {"superclass": SUPERCLASS_EPOCHS, "subclass": SUBCLASS_EPOCHS}})

## 4. Metadata and patient-split audit
Patient integrity always runs in load_ptbxl; RUN_METADATA_AUDIT controls the extra two-task summary/export.
Superclasses and 23 mapped subclasses retain multiple diagnoses and rare columns. Official folds remain unchanged.
Dataset release differences affect cohort/splits/counts: results from 1.0.1 and 1.0.3 must not be pooled as identical benchmarks.

In [ ]:
if ACTIVE and RUN_METADATA_AUDIT:
    with stage_log(SESSION_DIR, "metadata and patient-split audit"):
        AUDIT = audit_metadata(DATA_DIR, dataset_version=DATASET_VERSION)
        print(AUDIT["summary"], flush=True)
        write_json(SESSION_DIR / "metadata_audit.json", AUDIT["summary"])
        for task, table in AUDIT["statistics"].items():
            table.to_csv(SESSION_DIR / f"{task}_class_statistics.csv", index=False)
            print(task, "classes:", list(AUDIT["targets"][task].columns), flush=True)

## 5. Cached deterministic preprocessing and shared normalization
The first run creates one filtered float32 NumPy memory map, ~1.05 GB at 100 Hz for all ECGs.
This reads/filters each required waveform once, with progress; subsequent epochs read the local cache.
Random augmentation is never cached. Sampled normalization defaults to 1000 deterministic training-only ECGs;
full mode uses all training ECGs. Configuration, selected IDs, means/stds and seed are saved.
Keys include dataset metadata/raw-file stats and preprocessing settings; completed arrays are checksum-verified.
Both tasks share the cache/statistics. Diagnostics cache only required train/val/normalization records and never evaluate test.

In [ ]:
if ACTIVE and (RUN_SMOKE or RUN_SUPERCLASS or RUN_SUBCLASS):
    with stage_log(SESSION_DIR, "deterministic waveform cache and normalization"):
        tasks = [task for task, enabled in (("superclass", RUN_SUPERCLASS or RUN_SMOKE),
                                           ("subclass", RUN_SUBCLASS or RUN_SMOKE)) if enabled]
        for task in tasks:
            PREPARED[task] = load_ptbxl(task, config=DATA_CONFIG, seed=SEED,
                                        diagnostic_limits=diagnostic_limits(CONFIG))
            print(task, "ready; split counts:", {k: len(v) for k, v in PREPARED[task].splits.items()}, flush=True)
        write_json(SESSION_DIR / "normalization.json", {
            "statistics": PREPARED[tasks[0]].normalizer.state_dict(),
            "configuration": PREPARED[tasks[0]].provenance["normalization_configuration"]})
        if len(tasks) == 2:
            assert PREPARED[tasks[0]].normalizer.state_dict() == PREPARED[tasks[1]].normalizer.state_dict()

## 6. Lightweight real-data Mamba smoke
Uses cached train/validation batches and the same fitted normalization as training, no augmentation.
Each head gets one CUDA/AMP optimizer step, then a validation forward; no test tuning/evaluation.
Smoke models are discarded before fresh full models are created. RUN_SMOKE=False deliberately skips this extra
check; the dependency-level CUDA probe still runs. No interactive gate is required during Run All.

In [ ]:
if ACTIVE and RUN_SMOKE:
    with stage_log(SESSION_DIR, "real-data full-architecture Mamba smoke"):
        for task in ("superclass", "subclass"):
            model = None
            try:
                seed_everything(SEED)
                batch = cached_smoke_data(PREPARED[task], SMOKE_BATCH_SIZE)
                log_progress(f"Initializing reference Mamba {task} smoke")
                model = create_model("mamba", len(batch["class_names"]), CONFIG.model)
                SMOKE_REPORTS[task] = smoke_forward_backward(model, batch, device="cuda", mixed_precision=MIXED_PRECISION)
                write_json(SESSION_DIR / f"{task}_smoke.json", SMOKE_REPORTS[task])
                print("Smoke passed (not benchmark results):", task, SMOKE_REPORTS[task], flush=True)
            finally:
                del model
                gc.collect()
                torch.cuda.empty_cache()

## 7. Superclass Mamba training
Fresh model, all labeled training records in benchmark mode. First batch/step, batch progress, epoch metrics,
data-wait time and checkpoint writes are logged. Diagnostic limits are explicit and cannot produce benchmark results.
An optional original run directory resumes last.pth with unchanged config and best.pth present.

In [ ]:
def execute_task(task, epochs_override):
    try:
        task_config = replace(CONFIG, task=task, epochs=epochs_override if epochs_override is not None else FULL_EPOCHS)
        run_dir = RESUME_RUN_DIRS[task] or None
        if run_dir:
            # Restore the original config verbatim; overrides must match, never silently change the schedule.
            import json
            saved = json.loads((Path(run_dir) / "config.json").read_text())
            candidate = asdict(task_config)
            candidate["output_root"] = saved["output_root"]
            if candidate != saved:
                raise ValueError("Resume config differs from original; match initial settings including dataset/cache paths and epoch budget")
            task_config = replace(task_config, output_root=saved["output_root"])
        TASK_RESULTS[task] = run_experiment(config=task_config, prepared_data=PREPARED[task],
            run_dir=run_dir, resume_from=str(Path(run_dir) / "last.pth") if run_dir else None)
        # Retain summary/path, not unnecessary history frames for both experiments.
        print(task, TASK_RESULTS[task]["summary"], flush=True)
    except Exception as exc:
        TASK_ERRORS[task] = str(exc)
        print(f"FAILED {task}: {exc}; preserve its run.log/status.json and session log", flush=True)
        if not CONTINUE_OTHER_TASK_ON_FAILURE:
            raise
    finally:
        gc.collect()
        torch.cuda.empty_cache()

if ACTIVE and RUN_SUPERCLASS:
    with stage_log(SESSION_DIR, "superclass training/evaluation"):
        execute_task("superclass", SUPERCLASS_EPOCHS)

## 8. Subclass Mamba training — sequential after superclass, all 23 mapped labels

In [ ]:
if ACTIVE and RUN_SUBCLASS:
    with stage_log(SESSION_DIR, "subclass training/evaluation"):
        execute_task("subclass", SUBCLASS_EPOCHS)

## 9. Evaluation and result exports
run_experiment restores the best validation checkpoint, tunes thresholds on validation only, and evaluates test
only in benchmark mode. CSV/JSON predictions/metrics, config, environment, normalization/splits, history,
curves and checkpoints are saved per task. Completed artifacts survive another task's failure.

In [ ]:
if ACTIVE:
    with stage_log(SESSION_DIR, "export verification"):
        for task, result in TASK_RESULTS.items():
            directory = Path(result["output_dir"])
            required = ["best.pth", "last.pth", "config.json", "environment.json", "data_manifest.json",
                        "class_statistics.csv", "summary.json", "status.json", "run.log", "history.csv",
                        "figures/loss_curves.png", "figures/confusion_matrices.png"]
            splits = ("diagnostic_validation",) if DIAGNOSTIC_MODE else ("validation", "test", "test_at_0.5")
            required += [f"{split}/{file}" for split in splits for file in
                         ("metrics.json", "per_class.csv", "confusion_matrices.csv", "predictions.csv")]
            if not DIAGNOSTIC_MODE:
                required += ["thresholds.json"]
            missing = [file for file in required if not (directory / file).is_file()]
            if missing:
                TASK_ERRORS[task] = f"Incomplete exports: {missing}"
            print(task, "output:", directory, "missing:", missing, flush=True)

## 10. Final experiment summary
Save the entire session directory plus reusable mamba_wheels and cache as Kaggle outputs as space permits.
Upload saved wheels as a private input dataset for future fresh sessions; binary packages do not persist.
Cache holds no augmentation. Raw source-file size/mtime signatures detect normal waveform replacement;
for adversarial or timestamp-preserving edits, use a new cache directory or clear/rebuild the cache.
A failed task makes the final notebook fail after writing its summary, while successful-task artifacts remain.

In [ ]:
if ACTIVE:
    final_summary = {"declared_dataset_version": DATASET_VERSION, "dataset_root": str(DATA_DIR),
        "diagnostic_only": DIAGNOSTIC_MODE,
        "completed": {task: {"output_dir": r["output_dir"], "summary": r["summary"]} for task, r in TASK_RESULTS.items()},
        "errors": TASK_ERRORS, "session_dir": str(SESSION_DIR), "cache_dir": CACHE_DIR,
        "smoke_tasks": list(SMOKE_REPORTS)}
    write_json(SESSION_DIR / "final_summary.json", final_summary)
    write_json(SESSION_DIR / "pipeline_status.json", {"status": "failed" if TASK_ERRORS else "complete", "errors": TASK_ERRORS})
    print(final_summary, flush=True)
    if TASK_ERRORS:
        raise RuntimeError(f"Some tasks failed: {TASK_ERRORS}. Logs and completed artifacts are preserved at {SESSION_DIR}")
else:
    print("All execution flags are False; no installation, data reads, training or GPU validation occurred.")